In [21]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Annotated
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import operator

In [22]:
load_dotenv()

True

In [23]:
#we need to make sure that we are using correct model for structed output.beacuse we need exactly 2 output from all the nodes (

model= ChatGoogleGenerativeAI( model="gemini-2.5-flash")

In [24]:
#format of output structure from all the nodes should be same so that we can easily combine them in final node
#made using pydantic BaseModel class. we can use this class in all the nodes to make sure that output is in correct format.
class EvaluationSchema(BaseModel):
    score: int = Field(description="Score out of 10",ge=0,le=10)
    feedback: str = Field(description="detailed feedback for the essay")

In [25]:
structured_model = model.with_structured_output(EvaluationSchema)


In [26]:
class EssayState(TypedDict):
    essay: str
    cot_feedback: str  #clarity of thought feedback
    doa_feedback: str  #depth of analysis feedback
    lang_feedback: str  #language feedback
    summary: str #final feedback summary

    #we will use merge function to merge all feedbacks into a single list of individual scores 
    # as all three nodes will return parallelly so their is chang of overwriting hence use merge (reducer)
    individual_score: Annotated[list[int], operator.add]  #list of individual scores for each feedback
    average_score: float  #average score across all feedbacks


In [27]:
def eval_lang(state: EssayState) -> dict:
    #evaluate language of the essay
    prompt = f"Evaluate the language of the following essay and provide a score out of 10 and detailed feedback:\n{state['essay']}"
    output = structured_model.invoke(prompt)

    return{"lang_feedback": output.feedback, "individual_score": [output.score]}


In [28]:
def eval_doa(state: EssayState) -> dict:
    #evaluate depth of analysis of the essay
    prompt = f"Evaluate the depth of analysis of the following essay and provide a score out of 10 and detailed feedback:\n{state['essay']}"
    output = structured_model.invoke(prompt)

    return{"doa_feedback": output.feedback, "individual_score": [output.score]}

In [29]:
def eval_cot(state: EssayState) -> dict:
    #evaluate clarity of thought of the essay
    prompt = f"Evaluate the clarity of thought of the following essay and provide a score out of 10 and detailed feedback:\n{state['essay']}"
    output = structured_model.invoke(prompt)

    return{"cot_feedback": output.feedback, "individual_score": [output.score]}

In [30]:
def final_summary(state: EssayState) -> dict:
    #evaluate language of the essay
    prompt = f"Based on the following feedbacks create a summarised feedback \n language feedback - {state['lang_feedback']} \n depth of analysis feedback - {state['doa_feedback']} \n clarity of thought feedback - {state['cot_feedback']}"
    summary = model.invoke(prompt).content

    average_score = sum(state['individual_score'])/len(state['individual_score'])


    return{"summary": summary, "average_score": average_score}

In [31]:
graph = StateGraph(EssayState)

graph.add_node("eval_cot", eval_cot)
graph.add_node("eval_doa", eval_doa)
graph.add_node("eval_lang", eval_lang)
graph.add_node("final_summary", final_summary)

graph.add_edge(START, "eval_cot")
graph.add_edge(START, "eval_doa")       
graph.add_edge(START, "eval_lang")
graph.add_edge("eval_cot", "final_summary") 
graph.add_edge("eval_doa", "final_summary") 
graph.add_edge("eval_lang", "final_summary") 
graph.add_edge("final_summary", END)

workflow = graph.compile()


In [32]:
essay = """### Artificial Intelligence

Artificial Intelligence (AI) is a branch of computer science that enables machines to perform tasks that normally require human intelligence. These tasks include learning, problem-solving, decision-making, understanding language, and recognizing images or patterns.

AI is widely used in our daily lives. Virtual assistants, recommendation systems, self-driving technologies, medical diagnosis, chatbots, and fraud detection are some common examples. Machine learning and deep learning allow AI systems to learn from large amounts of data and improve their performance over time.

AI has the potential to transform many fields, including healthcare, education, transportation, agriculture, and business. However, it also creates challenges such as job displacement, privacy concerns, bias, and the responsible use of technology.

In conclusion, AI is a powerful technology that can greatly benefit society when developed and used responsibly. Its future depends on balancing innovation with human values, safety, and ethical considerations.
"""

In [35]:
essay_2 = """### Artifical Inteligence

AI is a very important thing in todays world and it is used in many places. AI means artifical inteligence which make computer think like human. It can do many works like answering questions, making images, driving cars and many more things.

AI is very helpfull for people because it save time and do work very fast. In hospitals it can help in finding deseases and in schools it can help students in study. Many apps are also using AI but sometimes it gives wrong answers.

AI is growing very fast and in future it will become more bigger. It can make our life easy but it can also create problems for peoples and jobs. So AI should be used carefully and we should not depend on it too much.

In the end, AI is a very usefull technology but we dont know how much it will change our future.
"""

In [36]:
initial_state = {
    "essay": essay_2
}
workflow.invoke(initial_state)

{'essay': '### Artifical Inteligence\n\nAI is a very important thing in todays world and it is used in many places. AI means artifical inteligence which make computer think like human. It can do many works like answering questions, making images, driving cars and many more things.\n\nAI is very helpfull for people because it save time and do work very fast. In hospitals it can help in finding deseases and in schools it can help students in study. Many apps are also using AI but sometimes it gives wrong answers.\n\nAI is growing very fast and in future it will become more bigger. It can make our life easy but it can also create problems for peoples and jobs. So AI should be used carefully and we should not depend on it too much.\n\nIn the end, AI is a very usefull technology but we dont know how much it will change our future.\n',
 'cot_feedback': "The essay attempts to cover various aspects of Artificial Intelligence, including its definition, applications, benefits, and potential draw